In [0]:

-- 1. Staging view for basic cleansing and normalization
CREATE STREAMING LIVE VIEW stg_stores AS
SELECT
  store_id,
  region_id,
  UPPER(store_type) AS store_type, -- Fix inconsistent casing
  store_name,
  store_street_address,
  store_city,
  store_state,
  store_country,
  store_phone, -- We keep null phone numbers as they are often realistically missing
  
  -- Handle date formatting anomalies
  COALESCE(
    try_to_date(first_opened_date, 'MM/dd/yyyy'),
    try_to_date(first_opened_date, 'yyyy-MM-dd'),
    try_to_date(first_opened_date, 'dd-MMM-yy')
  ) AS first_opened_date,
  
  last_remodel_date, -- Already a DATE type in schema
  total_sqft,
  grocery_sqft,
  ingested_timestamp,
  batch_date
FROM STREAM(mavenmarket.bronze.stores);

-- 2. Clean Silver Table filtering out illogical business metrics
CREATE STREAMING LIVE TABLE silver_stores (
  CONSTRAINT valid_sqft EXPECT (total_sqft > 0 AND grocery_sqft <= total_sqft) ON VIOLATION DROP ROW
)
COMMENT "Cleaned store dimension"
AS SELECT * FROM STREAM(live.stg_stores);